# Pipeline test for the KuKi LLM experiments

Runs every script end-to-end on **synthetic aggregate files** (same format as
`kuki_ru_aggregate.jsonl`) with **mock models** (random but schema-valid answers). No GPU needed.
Loading and splitting use exactly the same scripts as the real data. Everything is written to
`<repo>/test_run/`; real data in `<repo>/data/` is never touched.

Numbers produced here are meaningless; the test checks that files, joins and metrics work.
Always run in a **fresh kernel**, top to bottom. Real data and real models: `test_experiments.ipynb`.


In [1]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
import json
import os
import shutil
import sys

# Test mode must be switched on before config is imported; config reads it only once per kernel.
os.environ["KUKI_TEST"] = "1"
import config

if getattr(config, "TEST_MODE", None) is not True:  # re-running this cell is fine; mixing with real data is not
    raise RuntimeError(f"config loaded in the wrong mode or from the wrong place ({config.__file__}): restart the kernel")

assert config.BASE_DIR == config.ROOT / "test_run"  # never delete anything outside test_run
os.chdir(config.ROOT)  # so that %run finds the scripts
shutil.rmtree(config.BASE_DIR, ignore_errors=True)

# Dummy codebook prompts; the real wrappers.json is reused.
config.PROMPT_DIR.mkdir(parents=True)
shutil.copy(config.ROOT / "prompts" / "wrappers.json", config.PROMPT_DIR)
for layer in ["L1", "L2", "L3", "L4"]:
    for lang in ["en", "ru", "tr"]:
        (config.PROMPT_DIR / f"codebook_{layer}_{lang}.md").write_text(f"Test codebook {layer} ({lang})", encoding="utf-8")

[config] TEST data: /home/hau/kuki_llm/test_run/data


## 1. Synthetic aggregate file
Seven Russian articles in the format of `kuki_ru_aggregate.jsonl` (one record per article,
annotations keyed by Label Studio user id). Coverage mimics the block design: triple, double, single.


In [2]:
def make_content(i):
    return (f"Газ и безопасность {i}\n"
            "Москва угрожает Европе прекращением поставок газа.\n"
            "Жители Киева пострадали от обстрелов.\n"
            "Эксперты предупреждают о кризисе.")


def span(content, text, layer, label, **fields):
    start = content.index(text)
    return {"layer": layer, "label": label, "start": start, "end": start + len(text), "text": text, **fields}


def annotation(letter, content, i):
    # Even articles: all annotators agree on 'Economic' and a 'Call' span -> human alpha is defined.
    frames = ["Economic"] if i % 2 == 0 else []
    spans = [span(content, "о кризисе", "l3_persuasion", "Call")] if i % 2 == 0 else []
    if letter == "A":
        frames += ["Security & defense", "Political"]
        spans += [span(content, "Москва", "l1_roles", "ANTAGONIST"),
                  span(content, "Европе", "l1_roles", "INNOCENT"),
                  span(content, "угрожает Европе", "l3_persuasion", "Manipulative Wording"),
                  span(content, "прекращением поставок газа", "l4_coded", "Coded Phrase",
                       literal="stopping gas", insider="energy as weapon", why="context")]
    elif letter == "B":
        frames += ["Security & defense"]
        spans += [span(content, "Москва", "l1_roles", "ANTAGONIST"),
                  span(content, "Жители Киева", "l1_roles", "INNOCENT"),
                  span(content, "Эксперты предупреждают", "l3_persuasion", "Justification")]
    else:
        frames += ["Security & defense", "External regulation & reputation"]
        spans += [span(content, "Европе", "l1_roles", "INNOCENT"),
                  span(content, "пострадали от обстрелов", "l3_persuasion", "Manipulative Wording")]
    return {"frames": frames, "spans": spans}


COVERAGE = ["ABC", "ABC", "AB", "BC", "A", "B", "C"]  # annotators of each article
USER_IDS = {letter: user for user, letter in config.ANNOTATOR_IDS["ru"].items()}
SOURCES = ["ria_novosti", "theinsider"]

config.DATA_DIR.mkdir(parents=True)
with open(config.AGGREGATES["ru"], "w", encoding="utf-8") as f:
    for i, letters in enumerate(COVERAGE):
        content = make_content(i)
        rec = {"doc_id": f"{SOURCES[i % 2]}:{i}", "lang": "ru", "source": SOURCES[i % 2],
               "title": content.split("\n")[0], "content": content, "n_annotators": len(letters),
               "annotations": {str(USER_IDS[l]): annotation(l, content, i) for l in letters}}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


## 2. Structural check
The same check as in `test_experiments.ipynb`.


In [3]:
import json
from collections import Counter

from llm_io import read_jsonl


def inspect_aggregate(lang):
    """Structural check of an aggregate file: counts, users, span layers."""
    recs = read_jsonl(config.AGGREGATES[lang])
    anns = [a for r in recs for a in r["annotations"].values()]
    print(f"{config.AGGREGATES[lang].name}: {len(recs)} articles, {len(anns)} annotations")
    print("  articles by number of annotators:", dict(sorted(Counter(r["n_annotators"] for r in recs).items())))
    print("  users:", dict(Counter(u for r in recs for u in r["annotations"])), "expected:", config.ANNOTATOR_IDS[lang])
    print("  span layers:", dict(Counter(s["layer"] for a in anns for s in a["spans"])))
    print("  sources:", dict(Counter(r["source"] for r in recs)), "unknown:", {r["source"] for r in recs} - set(config.SOURCES))


inspect_aggregate("ru")


/home/hau/kuki_llm/venvkuki/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


kuki_ru_aggregate.jsonl: 7 articles, 13 annotations
  articles by number of annotators: {1: 3, 2: 2, 3: 2}
  users: {'8': 4, '4': 5, '10': 4} expected: {8: 'A', 4: 'B', 10: 'C'}
  span layers: {'l3_persuasion': 20, 'l1_roles': 22, 'l4_coded': 4}
  sources: {'ria_novosti': 4, 'theinsider': 3} unknown: set()


## 3. Data preparation and splits

In [4]:
%run prep_00_load_data.py --langs ru

articles per language x number of annotators:
n_annotators  1  2  3
lang                 
ru            3  2  2
human label rows: 599, L4 spans: 4
dropped 0 spans with a label from another layer:


In [5]:
from llm_io import read_jsonl

# Expected: the annotators per article as in COVERAGE.
for a in read_jsonl(config.PREP_DIR / "articles.jsonl"):
    print(a["article_id"], a["annotators"])


ria_novosti:0 ['ru_A', 'ru_B', 'ru_C']
theinsider:1 ['ru_A', 'ru_B', 'ru_C']
ria_novosti:2 ['ru_A', 'ru_B']
theinsider:3 ['ru_B', 'ru_C']
ria_novosti:4 ['ru_A']
theinsider:5 ['ru_B']
ria_novosti:6 ['ru_C']


In [6]:
import pandas as pd
import config
from llm_io import read_jsonl

articles = read_jsonl(config.PREP_DIR / "articles.jsonl")
print("entities of article 0:", articles[0]["entities"])
labels = pd.read_csv(config.PREP_DIR / "human_labels.csv")
# Expected: Москва/Европе/Жители Киева as entities; frames and paragraph labels per annotator.
labels[labels["value"] == 1].head(12)

entities of article 0: [{'key': 'европа', 'display': 'Европе'}, {'key': 'житель киев', 'display': 'Жители Киева'}, {'key': 'москва', 'display': 'Москва'}]


,lang,article_id,annotator,layer,unit,label,value
0,ru,ria_novosti:0,ru_A,L2,doc,Economic,1
7,ru,ria_novosti:0,ru_A,L2,doc,Security & defense,1
12,ru,ria_novosti:0,ru_A,L2,doc,Political,1
25,ru,ria_novosti:0,ru_A,L3,p1,Manipulative Wording,1
36,ru,ria_novosti:0,ru_A,L3,p3,Call,1
40,ru,ria_novosti:0,ru_A,L1,e:европа,INNOCENT,1
45,ru,ria_novosti:0,ru_A,L1,e:москва,ANTAGONIST,1
47,ru,ria_novosti:0,ru_B,L2,doc,Economic,1
54,ru,ria_novosti:0,ru_B,L2,doc,Security & defense,1
80,ru,ria_novosti:0,ru_B,L3,p3,Justification,1


In [7]:
%run prep_01_make_splits.py --n-dev 1 --n-l4 2

not annotated (excluded everywhere): 0
dev                  {'ru': 1}
s1_main_grid         {'ru': 4}
s2_stability         {'ru': 4}
s3_metadata_probe    {'ru': 4}
s4_generalisation    {'ru': 2}
s5_l4_pilot          {'ru': 2}


## 4. Prompt check
What the model actually sees: one paragraph-with-context call, native prompt language.

In [8]:
import llm_io

article = llm_io.load_split("s1_main_grid")[0]
for message in llm_io.build_messages(article, "L3", "para_ctx", "native", target=2):
    print(f"--- {message['role']} ---\n{message['content']}\n")
print(json.dumps(llm_io.output_schema("L1", len(article["entities"])), ensure_ascii=False)[:300])

--- system ---
Test codebook L3 (ru)

--- user ---
Полный текст статьи, только для контекста (абзацы пронумерованы [P0], [P1], [P2] и т. д.):

[P0] Газ и безопасность 0

[P1] Москва угрожает Европе прекращением поставок газа.

[P2] Жители Киева пострадали от обстрелов.

[P3] Эксперты предупреждают о кризисе.

Используя статью выше как контекст, разметьте только абзац [P2] в соответствии с кодбуком:

[P2] Жители Киева пострадали от обстрелов.

Отвечайте только в формате JSON в требуемой структуре.

{"type": "object", "required": ["roles"], "properties": {"roles": {"type": "array", "items": {"type": "object", "required": ["entity", "role"], "properties": {"entity": {"type": "string", "enum": ["E0", "E1", "E2"]}, "role": {"type": "string", "enum": ["PROTAGONIST", "ANTAGONIST", "INNOCENT"]}}}}}}


## 5. Stage 1: main grid (mock models)
Two mock models give two sizes and two profiles, so the decomposition has something to split.

In [9]:
%run s1_main_grid.py --model mock-M --dev
%run s1_main_grid.py --model mock-M
%run s1_main_grid.py --model mock-S

s1_dev__mock-M.jsonl: 54 jobs, 0 already done, 54 to run


s1_dev__mock-M: 100%|██████████| 54/54 [00:00<00:00, 3037.11call/s]


s1_main_grid__mock-M.jsonl: 216 jobs, 0 already done, 216 to run


s1_main_grid__mock-M: 100%|██████████| 216/216 [00:00<00:00, 12021.89call/s]


s1_main_grid__mock-S.jsonl: 216 jobs, 0 already done, 216 to run


s1_main_grid__mock-S: 100%|██████████| 216/216 [00:00<00:00, 12037.22call/s]


In [10]:
# Re-running must resume: expect '0 to run'.
%run s1_main_grid.py --model mock-S

s1_main_grid__mock-S.jsonl: 216 jobs, 216 already done, 0 to run


In [11]:
# One prediction file per experiment and model.
print(sorted(p.name for p in config.PRED_DIR.iterdir()))
predictions = pd.DataFrame(read_jsonl(config.PRED_DIR / "s1_main_grid__mock-M.jsonl") + read_jsonl(config.PRED_DIR / "s1_main_grid__mock-S.jsonl"))
print(predictions.groupby(["model", "layer", "granularity", "prompt_lang"]).size().unstack())
predictions[["key", "raw", "parse_ok"]].head(3)

['s1_dev__mock-M.jsonl', 's1_main_grid__mock-M.jsonl', 's1_main_grid__mock-S.jsonl']
prompt_lang               en  native
model  layer granularity            
mock-M L1    doc           4       4
             para         16      16
             para_ctx     16      16
       L2    doc           4       4
             para         16      16
             para_ctx     16      16
       L3    doc           4       4
             para         16      16
             para_ctx     16      16
mock-S L1    doc           4       4
             para         16      16
             para_ctx     16      16
       L2    doc           4       4
             para         16      16
             para_ctx     16      16
       L3    doc           4       4
             para         16      16
             para_ctx     16      16


,key,raw,parse_ok
0,s1_main_grid|mock-M|ria_novosti:0|L1|doc|en|no...,"{""roles"": [{""entity"": ""E2"", ""role"": ""ANTAGONIS...",True
1,s1_main_grid|mock-M|ria_novosti:0|L1|doc|nativ...,"{""roles"": [{""entity"": ""E2"", ""role"": ""ANTAGONIS...",True
2,s1_main_grid|mock-M|ria_novosti:0|L1|para|en|n...,"{""roles"": []}",True


In [12]:
%run evaluate.py --experiment s1_main_grid

     model lang layer granularity prompt_lang metadata      q  alpha_pair  persp_acc  parse_fail  error_rate
0   mock-M   ru    L1         doc          en     none  0.659       0.028      0.833         0.0         0.0
1   mock-M   ru    L1         doc      native     none  0.295      -0.058      0.861         0.0         0.0
2   mock-M   ru    L1        para          en     none -0.367      -0.203      0.667         0.0         0.0
3   mock-M   ru    L1        para      native     none  0.200      -0.068      0.722         0.0         0.0
4   mock-M   ru    L1    para_ctx          en     none -0.101      -0.095      0.667         0.0         0.0
5   mock-M   ru    L1    para_ctx      native     none  0.006      -0.173      0.611         0.0         0.0
6   mock-M   ru    L2         doc          en     none  0.304      -0.266      0.875         0.0         0.0
7   mock-M   ru    L2         doc      native     none  0.304      -0.162      0.786         0.0         0.0
8   mock-M   ru    

Expected here: a `SingularMatrixWarning`, because in the mock grid size and profile are confounded (mock-S = S + english, mock-M = M + multilingual). The real grid crosses them.

In [13]:
%run analyze_s1_decomposition.py

lang layer                          term  share  sum_sq  df      F  PR(>F)
  ru    L1 C(granularity):C(prompt_lang)  0.359   0.656 2.0  1.882   0.347
  ru    L1     C(profile):C(granularity)  0.209   0.382 2.0  1.097   0.477
  ru    L1                      Residual  0.191   0.348 2.0    NaN     NaN
  ru    L1                    C(profile)  0.088   0.161 1.0  0.923   0.438
  ru    L1        C(size):C(prompt_lang)  0.051   0.093 1.0  0.536   0.540
  ru    L1        C(size):C(granularity)  0.050   0.092 2.0  0.263   0.792
  ru    L1            C(size):C(profile)  0.043   0.078 1.0  0.448   0.572
  ru    L1     C(profile):C(prompt_lang)  0.006   0.011 1.0  0.066   0.821
  ru    L1                       C(size)  0.004   0.007 1.0  0.038   0.864
  ru    L1                C(prompt_lang) -0.000  -0.000 1.0 -0.000   1.000
  ru    L1                C(granularity) -0.000  -0.000 2.0 -0.000   1.000
  ru    L2 C(granularity):C(prompt_lang)  0.320   0.077 2.0 23.402   0.041
  ru    L2               

/home/hau/kuki_llm/analyze_s1_decomposition.py:27: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  table = anova_lm(smf.ols(formula, data=g).fit(), typ=2)
/home/hau/kuki_llm/analyze_s1_decomposition.py:27: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  table = anova_lm(smf.ols(formula, data=g).fit(), typ=2)
/home/hau/kuki_llm/analyze_s1_decomposition.py:27: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  table = anova_lm(smf.ols(formula, data=g).fit(), typ=2)


## 6. Stages 2-5 (mock models)

In [14]:
%run s2_stability.py --model mock-M --granularity para_ctx --prompt-lang en --n-samples 3
%run evaluate.py --experiment s2_stability

s2_stability__mock-M.jsonl: 144 jobs, 0 already done, 144 to run


s2_stability__mock-M: 100%|██████████| 144/144 [00:00<00:00, 11554.56call/s]


    model lang layer granularity prompt_lang metadata      q  alpha_pair  persp_acc  parse_fail  error_rate
0  mock-M   ru    L1    para_ctx          en     none  0.254      -0.085      0.611         0.0         0.0
1  mock-M   ru    L2    para_ctx          en     none  0.280      -0.206      0.536         0.0         0.0
2  mock-M   ru    L3    para_ctx          en     none  0.310      -0.099      0.771         0.0         0.0


In [15]:
%run s3_metadata_probe.py --model mock-M
%run evaluate.py --experiment s3_metadata_probe
pd.read_csv(config.RESULT_DIR / "s3_metadata_probe_prevalence_by_meta.csv").head()

s3_metadata_probe__mock-M.jsonl: 36 jobs, 0 already done, 36 to run


s3_metadata_probe__mock-M: 100%|██████████| 36/36 [00:00<00:00, 10320.21call/s]


    model lang layer granularity prompt_lang metadata      q  alpha_pair  persp_acc  parse_fail  error_rate
0  mock-M   ru    L1         doc          en     none  0.505       0.180      0.944         0.0         0.0
1  mock-M   ru    L1         doc          en  swapped -0.020      -0.029      0.806         0.0         0.0
2  mock-M   ru    L1         doc          en     true  0.192      -0.012      0.889         0.0         0.0
3  mock-M   ru    L2         doc          en     none  0.750      -0.163      0.839         0.0         0.0
4  mock-M   ru    L2         doc          en  swapped  0.304      -0.163      0.804         0.0         0.0
5  mock-M   ru    L2         doc          en     true  0.304      -0.252      0.821         0.0         0.0
6  mock-M   ru    L3         doc          en     none  0.441      -0.057      0.969         0.0         0.0
7  mock-M   ru    L3         doc          en  swapped  0.441      -0.013      0.917         0.0         0.0
8  mock-M   ru    L3        

,model,lang,layer,label,true_pole,none,swapped,true,shift_true_vs_none,shift_swapped_vs_true
0,mock-M,ru,L1,ANTAGONIST,gov_close,0.166667,0.333333,0.000000,-0.166667,0.333333
1,mock-M,ru,L1,ANTAGONIST,gov_distant,0.333333,0.000000,0.333333,0.000000,-0.333333
2,mock-M,ru,L1,INNOCENT,gov_close,0.166667,0.333333,0.166667,0.000000,0.166667
3,mock-M,ru,L1,INNOCENT,gov_distant,0.166667,0.333333,0.000000,-0.166667,0.333333
4,mock-M,ru,L1,PROTAGONIST,gov_close,0.000000,0.166667,0.166667,0.166667,0.000000


In [16]:
%run s4_generalisation.py --model mock-S --granularity doc --prompt-lang en
%run evaluate.py --experiment s4_generalisation

s4_generalisation__mock-S.jsonl: 6 jobs, 0 already done, 6 to run


s4_generalisation__mock-S: 100%|██████████| 6/6 [00:00<00:00, 7999.31call/s]


    model lang layer granularity prompt_lang metadata   q  alpha_pair  persp_acc  parse_fail  error_rate
0  mock-S   ru    L1         doc          en     none NaN         0.0      0.667         0.0         0.0
1  mock-S   ru    L2         doc          en     none NaN         0.0      0.786         0.0         0.0
2  mock-S   ru    L3         doc          en     none NaN         0.0      0.896         0.0         0.0


In [17]:
%run s5_l4_pilot.py --model mock-M
%run evaluate.py --experiment s5_l4_pilot

s5_l4_pilot__mock-M.jsonl: 4 jobs, 0 already done, 4 to run


s5_l4_pilot__mock-M: 100%|██████████| 4/4 [00:00<00:00, 3840.06call/s]

    model lang layer granularity prompt_lang metadata  recall  precision  n_human_paragraphs  n_flagged_paragraphs
0  mock-M   ru    L4         doc          en     none     0.0        0.0                   1                     2
1  mock-M   ru    L4         doc      native     none     1.0        0.5                   1                     2


## 7. Smoke test with a real model
Use a **fresh kernel without the test cells above**, after `prep_00`/`prep_01` ran on the real
data, on the GPU machine. Loads the model with transformers and runs one constrained call:
check that the output is valid JSON, and look at token count and time per call.

In [18]:
from tqdm.auto import tqdm

RUN_REAL = True
print("RUN_REAL is set to", RUN_REAL)
if RUN_REAL:
    import llm_io
    print("Loading model and running a smoke test on the first article of the dev split...")
    backend = llm_io.load_model("olmo3-7b")
    print("Model loaded. Running smoke test...")
    article = llm_io.load_split("dev")[0]
    jobs = list(llm_io.make_jobs("smoke", [article], "olmo3-7b", ["L2", "L3"], ["doc"], ["en"]))
    for job in tqdm(jobs, desc="Smoke test"):
        record = llm_io.call_llm(backend, job)
        tqdm.write(f"{job['layer']} {record['parsed']} | tokens in/out: {record['tokens_in']} {record['tokens_out']} "
                   f"| seconds: {record['seconds']} | error: {record['error']}")

RUN_REAL is set to True
Loading model and running a smoke test on the first article of the dev split...


Loading checkpoint shards: 100%|██████████| 3/3 [00:07<00:00,  2.55s/it]


Model loaded. Running smoke test...


Smoke test:  50%|█████     | 1/2 [00:04<00:04,  4.83s/it]

L2 {'frames': ['Policy prescription & evaluation', 'Political', 'Security & defense', 'Economic']} | tokens in/out: 147 42 | seconds: 4.83 | error: None


Smoke test: 100%|██████████| 2/2 [00:15<00:00,  7.57s/it]

L3 None | tokens in/out: 147 106 | seconds: 10.29 | error: None


In [19]:
import time, torch, transformers
from lmformatenforcer.integrations.transformers import build_token_enforcer_tokenizer_data
from transformers import AutoTokenizer, AutoModelForCausalLM
print(transformers.__version__, torch.cuda.is_available())

hf = "allenai/Olmo-3-7B-Instruct"
t = time.time(); tok = AutoTokenizer.from_pretrained(hf); print("tokenizer", time.time() - t)
t = time.time(); llm = AutoModelForCausalLM.from_pretrained(hf, dtype="auto", device_map="auto"); print("model", time.time() - t)
t = time.time(); data = build_token_enforcer_tokenizer_data(tok); print("enforcer", time.time() - t)

4.57.6 True
tokenizer 0.8837592601776123


Loading checkpoint shards: 100%|██████████| 3/3 [00:05<00:00,  1.83s/it]


model 6.0362489223480225
enforcer 4.433132648468018
